In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import *
from mtrain.example_dir.core import load_npz, ExampleDir, LD
from mtrain.seg import mapillary as mapi

In [ ]:
dirs = globL(
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/false_positives/wall"
    ),
    "*",
)

In [ ]:
from mtrain.neg_mask.crops import bbox_only_mask, get_region_crops, Bbox

# we have trash mask
# we have wall mask
# we want to go through all parts of trash mask
# find corresponding wall mask

def _get_individual_masks(mask):
    mask = mask.astype(np.uint8)
    bboxes = list(get_region_crops(mask))
    return [bbox_only_mask(mask, bb, 2000) for bb in bboxes]


def find_trash_wall_pairs(trash_mask, wall_mask):
    wmasks = _get_individual_masks(wall_mask)
    tmasks = _get_individual_masks(trash_mask)

    tm_and_wm = []
    for tm in tmasks:
        max_sum = 0
        mywall = None
        for wm in wmasks:
            inters = tm & wm
            cursum = inters.sum()
            if max_sum < cursum:
                max_sum = cursum
                mywall = wm
        if mywall is not None:
            tm_and_wm.append((tm, mywall))

    return tm_and_wm

In [ ]:
d = dirs[4]
edir = ExampleDir(d, {}, {})
img = LD(edir.image_path)
tmask = edir.get_trash_mask("md", "md")
mapi_pred = LD(edir.mapi_mask_path())
mapi_mask = mapi.get_mask(mapi_pred, mapi.Label.WALL)
show([img, OV(img, tmask), mapi_mask, tmask], (20, 20), 2, "off")

In [ ]:
# pairs = find_trash_wall_pairs(tmask, mapi_mask)
# show(it_chain(pairs))

In [ ]:
# mapi.show_seg_mask(mapi_pred)

Okay we have now got the wall and trash masks pairs.  
What is my criteria? Stuff at the bottom of the walls can generally not be counted. We want to know if it comes near the bottom end of the wall. The easiest way to find the bottom edge is to get a bbox, but that would be a rectangle, which sucks.   

for now, lets say we have the way to do it. then we have a the lower coordinates of a polygon for wall and for trash.  
We simply find the intersection amount with a changed wall mask, with decreased height of the polygon.  

40 pixels is the minimum size we find in md-masks. 100 is the maximum. if it is more than 100, it is a composite. So definitely above 100 would give me stuff which really just stuck on walls (if the thing starts 100 px above the wall end, it is highly possible that it is stuck to the wall itself.).   

The next is that it is fully inside the wall. These two seem good enough conditions.  

In [ ]:
# show(pairs[0])

In [ ]:
# cnt, hierarchy = cv2.findContours(pairs[0][1], cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
# zero = np.zeros(pairs[0][1].shape)
# plt.imshow(cv2.drawContours(zero, cnt, 0, 255))

In [ ]:
from mtrain.neg_mask.crops import get_largest_bbox
import cv2
import numpy as np


def shrink_bottom_only(rect, pixels_to_remove=100):
    # 1. Get the 4 corner points
    box = cv2.boxPoints(rect)
    box = np.array(box, dtype="float32")

    # 2. Find the two "bottom" points (highest y-coordinates)
    # Sort points by their Y-coordinate (descending)
    indices = np.argsort(box[:, 1])[::-1]
    bottom_indices = indices[:2]
    top_indices = indices[2:]

    # 3. Calculate the direction vector of the side we are shrinking
    # We move from a bottom point toward its corresponding top point
    # We'll use the vector between the highest point and the point most "above" it
    p_bottom = box[bottom_indices[0]]

    # Find which top point is on the same side as p_bottom
    # (Checking distance to find the connected neighbor)
    dist1 = np.linalg.norm(p_bottom - box[top_indices[0]])
    dist2 = np.linalg.norm(p_bottom - box[top_indices[1]])

    p_top = box[top_indices[0]] if dist1 < dist2 else box[top_indices[1]]

    # Create a unit vector pointing from bottom to top
    vector = p_top - p_bottom
    length = np.linalg.norm(vector)

    if length == 0:
        return np.intp(box)

    unit_vector = vector / length

    # 4. Move both bottom points along that unit vector
    # This slides the "bottom bar" up toward the top
    box[bottom_indices[0]] += unit_vector * pixels_to_remove
    box[bottom_indices[1]] += unit_vector * pixels_to_remove

    return np.intp(box)


def decrease_wall_height_in_pair(pair):
    """given a trash mask and wall mask, we decrease the height of the wall mask at the bottom

    The intent is to find all detected trash stuff which is actually just a part of the wall
    examples include wall paintings, walls with stickers / posters, fungal walls, etc

    mapillary does not give perfect wall masks, otherwise i would simply cut it out 
    a lot of trash can be found at the edge of the photo, near the walls (it tends to go that way)
    the intuition is that it lies at the bottom of the wall

    md model does not technically create single object regions of height > 100px
    so if any trash detection is 100px the bottom of the wall it is attached to, then it is not trash

    we assume we are given a mask of a single trash object, and the mask of the wall it is surrounded by
    these are assumed to be single region masks

    - We first find the bbox of the trash object. 
    - from the wall mask, we are only interested in the part of the wall horizontally surrounding the trash object
    - so we create a mask from the bbox of the trash object, where the horizontal length is untouched
      - but the vertical length goes from 0 -> max mask shape
    - then we take an intersection of the wall mask with this mask 
      - we have the surrounding wall
    - now use cv2.contours + cv2.minAreaRect to get the wall bbox (this gives a rotated rectangle). 
      - we use rotated rectangles because walls in a street image are not horizontally straight, perspective makes them look rotated
    - we decrease the height of this bbox by cutting 100px from the bottom
    - we return final wall mask with this height cut (it is also hiorizontally cut now)
    - this returns the pair, trash mask is returned as is with the new wall mask
    """

    trash_mask, wall_mask = pair
    # bbox = get_largest_bbox(trash_mask)

    # full_rect_along_roi_length = cv2.rectangle(
    #     np.zeros(trash_mask.shape), (bbox.x, 0), (bbox.x2, trash_mask.shape[0]), 255, -1
    # )

    # # only the wall mask surrounding trash object horizontally
    # wall_roi_mask = (
    #     wall_mask.astype(bool) & full_rect_along_roi_length.astype(bool)
    # ).astype(np.uint8)

    cnt, hierarchy = cv2.findContours(
        wall_mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE
    )
    rect = cv2.minAreaRect(cnt[0])
    box = shrink_bottom_only(rect, 100)
    boxed = cv2.drawContours(np.zeros(wall_mask.shape), [box], 0, 255, -1)

    return (pair[0], boxed.astype(bool) & wall_mask.astype(bool))


def make_wall_mask_surround_only_trash_region(pair):
    trash_mask, wall_mask = pair
    bbox = get_largest_bbox(trash_mask)

    full_rect_along_roi_length = cv2.rectangle(
        np.zeros(trash_mask.shape), (bbox.x, 0), (bbox.x2, trash_mask.shape[0]), 255, -1
    )

    # only the wall mask surrounding trash object horizontally
    wall_roi_mask = (
        wall_mask.astype(bool) & full_rect_along_roi_length.astype(bool)
    ).astype(np.uint8)
    return (trash_mask, wall_roi_mask)


def do_trash_and_wall_intersect(pair):
    mask, wallmask = pair
    return (mask & wallmask).sum() > 0


def get_trash_masks_which_are_part_of_wall(full_trash_mask, full_wall_mask):
    part_of_wall_masks = []
    pairs = find_trash_wall_pairs(full_trash_mask, full_wall_mask)
    for pair in pairs:
        pair = make_wall_mask_surround_only_trash_region(pair)
        pair = decrease_wall_height_in_pair(pair)
        # part_of_wall_masks.append(pair)

        if do_trash_and_wall_intersect(pair):
            part_of_wall_masks.append(pair[0].astype(bool))
    return part_of_wall_masks

In [ ]:
d = dirs[17]
edir = ExampleDir(d, {}, {})
img = LD(edir.image_path)
tmask = edir.get_trash_mask("md", "md")
mapi_pred = LD(edir.mapi_mask_path())
mapi_mask = mapi.get_mask(mapi_pred, mapi.Label.WALL)

masks = get_trash_masks_which_are_part_of_wall(tmask, mapi_mask)
print("length ", len(masks))
show([img, OV(img, tmask), OV(img, mapi_mask)], (20, 20), 3, "off")
if masks:
    print(len(masks))
    show(masks)

now we have a function which give sus mask examples to use with the neg trash model.  
We get a list of single region masks. we need to now get padded crops around them adn create a dataset

In [ ]:
from mtrain.neg_mask.model.datasets.foviate_shrink import get_foviate_remaps, get_foviated_image_and_mask
from mtrain.neg_mask.crops import padded_crop
from pathlib import Path
from mtrain.utils import *
from mtrain.random import random_filename

WALL_DS = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean/walls_foveated")

mkdir(WALL_DS / "train")
mkdir(WALL_DS / "masks")

def save_mask_to_wall_dataset(image, mask, src_dir_name):
    bbox = get_largest_bbox(mask)
    _, (re_img, re_mask) = get_foviated_image_and_mask(image, mask.astype(np.uint8), bbox, 1024, 224, 10)
    fname = random_filename(8)
    fname = f'other_wall_{src_dir_name}_{fname}'
    DiskImage.save(re_img, WALL_DS / "train" / f"{fname}.jpg")
    DiskBooleanMask.save(re_mask, WALL_DS / "masks" / f"{fname}.png")

In [ ]:
from mtrain.neg_mask.ipywidgets.widget_10 import get_trash_mask
from tqdm import tqdm
from mtrain.seg import mapillary as mapi

dirs = globL(
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/false_positives/wall"
    ),
    "*",
)
dirs = [d for d in dirs if (d / "image.jpg").exists()]
edirs = [ExampleDir(d, {}, {}) for d in dirs]

for edir in tqdm(edirs):
    try:
        trash_mask = edir.get_trash_mask("md", "md")
        mapi_pred = DiskBooleanMask.load(edir.mapi_mask_path())
        mapi_mask = mapi.get_mask(mapi_pred, mapi.Label.WALL)
    except:
        print(f"WARN: could not get trash mask for {edir.d}")
        continue

    image = DiskImage.load(edir.image_path)
    try:
        masks = get_trash_masks_which_are_part_of_wall(trash_mask, mapi_mask)
        for mask in masks:
            try:
                save_mask_to_wall_dataset(image, mask, edir.d.name)
            except Exception as ex:
                print(f"WARN: {edir.d}: failure in saving {ex}")
    except Exception as ex:
        print(f"WARN: {edir.d} failure in getting trash masks {ex}")
        continue
    # masks = get_trash_masks_which_are_part_of_wall(trash_mask, mapi_mask)
    # for mask in masks:
    #     save_mask_to_wall_dataset(image, mask, edir.d.name)

In [ ]:
! ls "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/false_positives/wall" | wc -l

In [ ]:
from mtrain.utils import show_negmask_ds

images, masks = show_negmask_ds(WALL_DS)

In [ ]:
def find_in_fp_dir(image_path):
    name = Path(image_path).name[11:].split("_")[0]
    FP = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/inference/false_positives/wall")
    d = FP / name
    if not d.exists():
        print("no exist", d)
        return None
    edir = ExampleDir(d, {}, {})
    return edir.load_all_assets("md", "md")

In [ ]:
assets = find_in_fp_dir(images[8])

In [ ]:
show([
    assets["image"],
    OV(assets["image"], mapi.get_mask(assets["mapi_pred"], mapi.Label.WALL)),
    OV(assets["image"], assets["trash_mask"]),
], (20,20), ncols=3)

In [ ]:
wall_mask = mapi.get_mask(assets["mapi_pred"], mapi.Label.WALL)
masks = get_trash_masks_which_are_part_of_wall(assets["trash_mask"], wall_mask)

In [ ]:
wmasks =_get_individual_masks(wall_mask)

In [ ]:
show(wmasks)

In [ ]:
pairs = find_trash_wall_pairs(assets["trash_mask"], wall_mask)
show(it_chain(pairs))

In [ ]:
from mtrain.neg_mask.model.datasets.copy_ds import copy_negmask_ds_to_ds

In [ ]:
! ls {WALL_DS}/train | head

In [ ]:

FOVEATED_PATH = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/blurred/clean/foveated"
)
copy_negmask_ds_to_ds(WALL_DS, FOVEATED_PATH)

In [ ]:
# part_of_wall_masks = []
# pairs = find_trash_wall_pairs(full_trash_mask, full_wall_mask)
# for pair in pairs:
#     pair = make_wall_mask_surround_only_trash_region(pair)
#     pair = decrease_wall_height_in_pair(pair)
#     # part_of_wall_masks.append(pair)

#     if do_trash_and_wall_intersect(pair):
#         part_of_wall_masks.append(pair[0].astype(bool))
# return part_of_wall_masks
show(masks)

In [ ]:
from mtrain.example_dir.defaults.smallnet import default_smallnet_learners
from mtrain.seg import mapillary as mapi
MODELS_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models")
sml = default_smallnet_learners(MODELS_DIR, "md", 4)

sml["md"].strides = []
msk = sml["md"].predict(assets["image"])
gridded = draw_grid_cv2(assets["image"].copy(), 100)
show([gridded, OV(assets["image"], msk)])

In [ ]:
masks[0].shape